# Lens Aberration Simulator

Interactive simulation of a thin lens using the **lensmaker's equation** and the **Sellmeier dispersion formula**. We trace paraxial rays for three wavelengths (red C=656 nm, green e=546 nm, blue F=486 nm) and visualise where they focus — the essence of longitudinal chromatic aberration.

We also show the **lateral CA** at a given off-axis field angle.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'serif'

## 1. Sellmeier Dispersion

In [ ]:
SELLMEIER = {
    'N-BK7':  {'B': [1.03961212, 0.23179234, 1.01046945],
               'C': [6.00069867e-3, 2.00179144e-2, 1.03560653e2]},
    'N-F2':   {'B': [1.34533359, 0.20979017, 0.93397479],
               'C': [9.97743871e-3, 4.70450767e-2, 1.11886764e2]},
    'N-SF11': {'B': [1.73759695, 0.31395246, 1.18894207],
               'C': [1.31887070e-2, 6.23068142e-2, 1.55236290e2]},
    'N-FK5':  {'B': [0.84433418, 0.34172993, 0.92819083],
               'C': [4.73791365e-3, 1.49296800e-2, 9.72485246e1]},
}

def n(glass, lam_nm):
    lam2 = (lam_nm / 1000.0) ** 2
    c = SELLMEIER[glass]
    n2 = 1.0 + sum(b * lam2 / (lam2 - ci) for b, ci in zip(c['B'], c['C']))
    return np.sqrt(n2)

def thin_lens_f(glass, R1, R2, lam_nm):
    """Focal length of a thin biconvex lens at a given wavelength."""
    return 1.0 / ((n(glass, lam_nm) - 1.0) * (1.0/R1 - 1.0/R2))

## 2. Paraxial Ray Trace — Longitudinal CA

In [ ]:
def simulate_longitudinal_ca(glass='N-BK7', R1=61.5, R2=-61.5,
                              n_rays=5, object_distance=1000.0):
    """
    Trace marginal rays for R, G, B wavelengths and show where they focus.
    """
    wavelengths = {'Red (656 nm)': 656.3, 'Green (546 nm)': 546.1, 'Blue (486 nm)': 486.1}
    colors_plot  = {'Red (656 nm)': '#FF4444', 'Green (546 nm)': '#44BB44', 'Blue (486 nm)': '#4444FF'}

    fig, ax = plt.subplots(figsize=(12, 5))

    f_ref = thin_lens_f(glass, R1, R2, 546.1)  # green focal length

    focus_points = {}
    for name, lam in wavelengths.items():
        f = thin_lens_f(glass, R1, R2, lam)
        color = colors_plot[name]

        # Image distance from thin lens equation: 1/v = 1/f - 1/u  (u < 0)
        u = -abs(object_distance)
        v = 1.0 / (1.0/f - 1.0/u)
        focus_points[name] = v

        # Draw rays from object plane to lens (at x=0) then to focus
        heights = np.linspace(-R1*0.4, R1*0.4, n_rays)
        for h in heights:
            # Ray from object (x=-|u|) at height h*object/u, through lens at height h
            obj_h = h * (abs(u) / abs(u))  # parallel approximation
            ax.plot([-abs(u)*0.15, 0, v], [h, h, 0], color=color, alpha=0.5, lw=0.8)

        # Mark focus
        ax.axvline(v, color=color, ls='--', lw=0.8, alpha=0.7)
        ax.scatter([v], [0], color=color, s=40, zorder=5)

    # Draw lens
    ax.axvline(0, color='black', lw=2)
    ax.axhline(0, color='gray', lw=0.5, ls=':')

    ax.set_xlabel('Axial distance from lens (mm)')
    ax.set_ylabel('Ray height (mm)')
    ax.set_title(f'Longitudinal CA — {glass} biconvex lens (R₁={R1}, R₂={R2} mm)')
    ax.set_xlim(-abs(u)*0.18, max(focus_points.values()) * 1.08)
    ax.set_ylim(-R1*0.55, R1*0.55)

    patches = [mpatches.Patch(color=colors_plot[n], label=f'{n}  f={focus_points[n]:.2f} mm')
               for n in wavelengths]
    ax.legend(handles=patches, fontsize=9, loc='upper right')
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

    # Summary
    print(f'\nLongitudinal CA (focus spread):')
    f_red  = focus_points['Red (656 nm)']
    f_blue = focus_points['Blue (486 nm)']
    print(f'  Red focal distance : {f_red:.4f} mm')
    print(f'  Blue focal distance: {f_blue:.4f} mm')
    print(f'  Δf (R−B)           : {f_red - f_blue:.4f} mm')

simulate_longitudinal_ca('N-BK7', 61.5, -61.5)

## 3. Compare Glass Types

In [ ]:
wavelengths_nm = np.linspace(400, 700, 200)
R1, R2 = 61.5, -61.5

fig, ax = plt.subplots(figsize=(10, 5))
glass_colors = {'N-BK7': '#4477AA', 'N-F2': '#EE6677', 'N-SF11': '#AA3377', 'N-FK5': '#228833'}

for glass, color in glass_colors.items():
    f_vals = [thin_lens_f(glass, R1, R2, lam) for lam in wavelengths_nm]
    f_green = thin_lens_f(glass, R1, R2, 546.1)
    delta_f = np.array(f_vals) - f_green
    ax.plot(wavelengths_nm, delta_f, color=color, lw=2, label=glass)

ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Δf from green (mm)')
ax.set_title('Chromatic focal shift Δf(λ) by glass type')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Lateral CA at Field Angle

For an off-axis point at field angle ω, the lateral CA (image height difference) is:
$$\delta y' = y'_d \cdot \frac{f_d - f(\lambda)}{f(\lambda)} \approx -\frac{y'_d}{V}$$

In [ ]:
def lateral_ca_profile(glass, R1, R2, sensor_half_width_mm=18.0, n_points=100):
    """
    Return (heights_norm, ca_r_mm, ca_b_mm) arrays for a thin lens.
    heights_norm in [0, 1], CA in mm.
    """
    f_g = thin_lens_f(glass, R1, R2, 546.1)
    f_r = thin_lens_f(glass, R1, R2, 656.3)
    f_b = thin_lens_f(glass, R1, R2, 486.1)

    h_norm = np.linspace(0, 1, n_points)
    h_mm   = h_norm * sensor_half_width_mm

    # Lateral CA = h * (f_ref - f(λ)) / f_ref
    ca_r = h_mm * (f_g - f_r) / f_g
    ca_b = h_mm * (f_g - f_b) / f_g
    return h_norm, ca_r, ca_b

fig, ax = plt.subplots(figsize=(8, 5))
for glass, color in glass_colors.items():
    h, cr, cb = lateral_ca_profile(glass, 61.5, -61.5)
    ax.plot(h, cr*1000, color=color, ls='-',  lw=2, label=f'{glass} R−G')
    ax.plot(h, cb*1000, color=color, ls='--', lw=2, label=f'{glass} B−G')

ax.set_xlabel('Normalised image height')
ax.set_ylabel('Lateral CA (µm)')
ax.set_title('Lateral chromatic aberration vs image height')
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()